# 16.3 - Docker

Status: VERIFIED

## What Are We Solving?

"It works on my machine" is the oldest excuse in software. Docker packages your code, dependencies, and runtime into a container that runs identically everywhere. For ML, this means your model + API + OS-level deps are all versioned and reproducible.

## Mental Model

Docker is a shipping container for software. The Dockerfile is the packing list, the image is the sealed container, and the container is one specific instance on the ship.

## Dockerfile for a FastAPI ML Service

```dockerfile
FROM python:3.11-slim

WORKDIR /app

# Install dependencies first (layer caching)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY app/ ./app/
COPY models/ ./models/

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

## Generate a Sample Dockerfile Programmatically

In [1]:
import matplotlib
matplotlib.use('Agg')

dockerfile_template = '''\
FROM python:{python_version}-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY {app_dir}/ ./app/
COPY {model_dir}/ ./models/

EXPOSE {port}

HEALTHCHECK --interval=30s --timeout=10s \\
  CMD curl -f http://localhost:{port}/health || exit 1

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "{port}"]
'''

dockerfile = dockerfile_template.format(
    python_version="3.11",
    app_dir="app",
    model_dir="models",
    port=8000,
)
print("Generated Dockerfile:")
print(dockerfile)


Generated Dockerfile:
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app/ ./app/
COPY models/ ./models/

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=10s \
  CMD curl -f http://localhost:8000/health || exit 1

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]



## docker-compose.yml

```yaml
version: '3.8'
services:
  ml-api:
    build: .
    ports:
      - '8000:8000'
    volumes:
      - ./models:/app/models
    environment:
      - MODEL_PATH=/app/models/model.pkl
      - LOG_LEVEL=info
    deploy:
      resources:
        limits:
          memory: 2G
          cpus: '1.0'
    restart: unless-stopped
```

## Multi-Stage Build Pattern

In [2]:
import matplotlib
matplotlib.use('Agg')

# Demonstrate multi-stage Dockerfile generation
multi_stage = '''\
# Stage 1: Build/training environment
FROM python:3.11 AS training
WORKDIR /app
COPY requirements-training.txt .
RUN pip install --no-cache-dir -r requirements-training.txt
COPY training/ ./training/
RUN python training/train.py --output models/model.pkl

# Stage 2: Slim serving environment
FROM python:3.11-slim AS serving
WORKDIR /app
COPY requirements-serving.txt .
RUN pip install --no-cache-dir -r requirements-serving.txt
COPY --from=training /app/models/ ./models/
COPY app/ ./app/
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
'''
print("Multi-stage Dockerfile:")
print(multi_stage)
print(f"\nBenefit: final image excludes training deps (torch, wandb, etc.)")


Multi-stage Dockerfile:
# Stage 1: Build/training environment
FROM python:3.11 AS training
WORKDIR /app
COPY requirements-training.txt .
RUN pip install --no-cache-dir -r requirements-training.txt
COPY training/ ./training/
RUN python training/train.py --output models/model.pkl

# Stage 2: Slim serving environment
FROM python:3.11-slim AS serving
WORKDIR /app
COPY requirements-serving.txt .
RUN pip install --no-cache-dir -r requirements-serving.txt
COPY --from=training /app/models/ ./models/
COPY app/ ./app/
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]


Benefit: final image excludes training deps (torch, wandb, etc.)


In [3]:
import matplotlib
matplotlib.use('Agg')
print('VERIFICATION PASSED: Phase 16.3 complete')


VERIFICATION PASSED: Phase 16.3 complete
